# 6. Persona (sociodemographic) prompting

The model answers as a member of each group. A stand-in model plays the stereotype of the group (`right`, `woman` and `old` are always toxic) so that the effect is visible.

In [1]:
import tempfile
from pathlib import Path

import pandas as pd

from perspective_audit import annotate, load_model_inputs, run_dataset, save_au_tables
from perspective_audit.toy import toy_ratings, toy_spec

pd.set_option("display.width", 160)
result = run_dataset(toy_spec(), raw=toy_ratings())                      # the toy dataset
out = Path(tempfile.mkdtemp())                                           # where the runs are saved
save_au_tables({"toy": result.au}, out)                                  # aposteriori_for_models.csv: the attributed texts
inputs = load_model_inputs(out / "aposteriori_for_models.csv")
WORD = {"toy": "comment"}                                                # how a text is called in a prompt

## The default run first

The change under the persona is measured against the default prompt, so the default run must exist.

In [2]:
from perspective_audit.toy import standin_model

annotate(standin_model("standin"), inputs, out, prompt=lambda texts, item: f"Rate each {item} from 1 to 5:\n{texts}", run="baseline", item_words=WORD, show_progress=False)

,texts,batches,resumed,model calls,seconds
dataset,,,,,
toy,40,4,0,4,0.0


## The persona prompt

`load_persona_inputs` gives one row per text and characteristic, with the groups to answer as.

In [3]:
from perspective_audit import annotate_personas, load_persona_inputs
from perspective_audit.toy import standin_persona_model

persona_inputs = load_persona_inputs(out / "aposteriori_for_models.csv")
persona_inputs["toy"].head(3)

,item_id,text,dimension,groups
0,t2,Toy comment number 2.,gender,"[man, woman]"
1,t5,Toy comment number 5!,gender,"[man, woman]"
2,t8,Toy comment number 8.,gender,"[man, woman]"


In [4]:
def persona_prompt(text, item, dimension, groups):
    return f"""Answer as an annotator of each group, rating how toxic the {item} is from 1 to 5.
Groups: {", ".join(groups)}
Reply with JSON, one key per group, for example {{"{groups[0]}": {{"score": 3}}}}

{item}: {text}"""

annotate_personas(standin_persona_model("standin"), persona_inputs, out, prompt=persona_prompt, run="scd", seed=42, item_words=WORD, show_progress=False)

,texts,resumed,model calls,seconds
dataset,,,,
toy,40,0,40,0.0


## What was saved

One row per text, characteristic and group.

In [5]:
from perspective_audit import collect_persona_runs

long = collect_persona_runs(out, "scd")
long.head(6)

,dataset,item_id,dimension,text,model,run,group,position,score,label
0,toy,t2,gender,Toy comment number 2.,standin,scd,man,1,1,civil
1,toy,t2,gender,Toy comment number 2.,standin,scd,woman,2,5,toxic
2,toy,t5,gender,Toy comment number 5!,standin,scd,man,1,5,toxic
3,toy,t5,gender,Toy comment number 5!,standin,scd,woman,2,5,toxic
4,toy,t8,gender,Toy comment number 8.,standin,scd,woman,1,5,toxic
5,toy,t8,gender,Toy comment number 8.,standin,scd,man,2,1,civil


## Kappa under the persona prompt, and the change from the default

A kappa of 0 with `constant_model` means the model gave one label to every text of that group (it only played the stereotype).

In [6]:
from perspective_audit import compute_kappa

given = pd.read_csv(out / "aposteriori_for_models.csv", dtype={"item_id": str})
kappa = compute_kappa({"toy": result}, out, given, reps=1_000, seed=42, models=["standin"], prompt="scd")
kappa["by_group"][["characteristic", "group", "n_texts", "kappa", "ci_low", "ci_high", "constant_model"]]

,characteristic,group,n_texts,kappa,ci_low,ci_high,constant_model
0,gender,man,20,-1.0,-1.0,-0.6,False
1,gender,woman,20,0.0,0.0,0.0,True
2,politics,left,20,-1.0,-1.0,-0.6,False
3,politics,right,20,0.0,0.0,0.0,True


In [7]:
kappa["prompt_change"][["characteristic", "group", "kappa_baseline", "kappa_scd", "change"]]

,characteristic,group,kappa_baseline,kappa_scd,change
0,gender,man,-1.0,-1.0,0.0
1,gender,woman,1.0,0.0,-1.0
2,politics,left,-1.0,-1.0,0.0
3,politics,right,1.0,0.0,-1.0
